In [1]:
from pathlib import Path
from dateutil.parser import parse as parse_date

## 1. Select InSAR pairs with ASF Search

In [2]:
# # Define a project name
project_name = 'kathmandu_asc_2018_2020_10by2'
work_dir = Path.cwd()
data_dir = work_dir / 'kathmandu_asc_2018_2020_10by2'

# start_date = '2015-04-01'
# end_date = '2024-04-01'
# stack_start = parse_date(start_date + ' 00:00:00Z')
# stack_end = parse_date(end_date + ' 00:00:00Z')
# max_temporal_baseline = 36 #days
# year_start = stack_start.year
# year_end = stack_end.year
# # season_period = [start_date[5:], end_date[5:]]

data_dir.mkdir(parents=True, exist_ok=True)

# print(type(stack_start), stack_end)
print(Path.cwd())

/home/jovyan


In [3]:
# import asf_search as asf
# import pandas as pd

# # wkt = 'POLYGON((-135.7 58.2,-136.6 58.1,-135.8 56.9,-134.6 56.1,-134.9 58.0,-135.7 58.2))'
# # wkt = 'POINT(-107.3 75.3)'
# wkt = 'POLYGON((109.7803 36.8276,109.7869 36.7992,109.8394 36.7597,109.8486 36.7654,109.8437 36.7767,109.8142 36.8029,109.7901 36.8258,109.7803 36.8276))'   # Get from ASF AOI

# # All options to geo_search() can be specified using kwargs, which also allows them to be handled using a dictionary.
# opts = {
#     'platform': asf.SENTINEL1,
#     'start': start_date,
#     'end': end_date,
#     # 'season': '[152, 300]',
#     'processingLevel': asf.SLC,
#     'beamMode': asf.IW,
#     'flightDirection': asf.ASCENDING,
#     'relativeOrbit': 11
# }
# search_results = asf.geo_search(
#         intersectsWith=wkt,
#         **opts
#     )

In [4]:
# centroid = search_results[1].centroid().wkt
# print(centroid)
# centroid_results = asf.geo_search(intersectsWith=centroid, **opts)
# search_results = centroid_results
# print(f'{len(search_results)} SLC images found')
# # print(search_results[0])

In [5]:
# baseline_results = asf.baseline_search.stack_from_product(search_results[-1])

# columns = list(baseline_results[0].properties.keys()) + ['geometry', ]
# data = [list(scene.properties.values()) + [scene.geometry, ] for scene in baseline_results]

# stack = pd.DataFrame(data, columns=columns)
# stack['startTime'] = stack.startTime.apply(parse_date)

# stack = stack.loc[(stack_start <= stack.startTime) & (stack.startTime <= stack_end)]

In [6]:
# print(len(stack))

In [7]:
#Select the subset of seasonal search
# print(stack.columns)
# print(stack['startTime'])

# stack_season = pd.DataFrame()
# for year_index in range(int(year_start), int(year_end) + 1):
#     season_start = parse_date(f'{str(year_index)}-{season_period[0]} 00:00:00Z')
#     season_end = parse_date(f'{str(year_index)}-{season_period[1]} 00:00:00Z')
#     stack_season = pd.concat([stack_season, stack.loc[(season_start <= stack.startTime) & (stack.startTime <= season_end)]])
# print('Total SAR sences are:', len(stack_season.startTime))
# for i in stack_season.startTime.values:   # or stack_season['startTime'].values
#     print(i, type(i))

In [8]:
# I modified the code for seasonal search and inter-annual pairs.
# sbas_pairs = set()
# interannual_baseline_max = 385
# interannual_baseline_min = 345
# # Can modify the codes for different networking strategies.
# for reference, rt in stack_season.loc[::-1, ['sceneName', 'temporalBaseline']].itertuples(index=False):
#     # print(reference, rt)
#     secondaries = stack_season.loc[
#         (stack_season.sceneName != reference)
#         & (stack_season.temporalBaseline - rt <= interannual_baseline_max)
#         & (stack_season.temporalBaseline - rt > interannual_baseline_min)
#     ]
#     for secondary in secondaries.sceneName:
#         sbas_pairs.add((reference, secondary))
#         print(reference[17:25], secondary[17:25])
# print('The total number of interferometric pairs are: ', len(sbas_pairs))
# # print(sbas_pairs)

In [9]:
# # For adjacent interferogram pair search

# sbas_pairs = set()

# # In this case, only add the interferograms in 2023
# for reference, rt in stack.loc[::-1, ['sceneName', 'temporalBaseline']].itertuples(index=False):
#     secondaries = stack.loc[
#         (stack.sceneName != reference)
#         & (stack.temporalBaseline - rt <= max_temporal_baseline)
#         & (stack.temporalBaseline - rt > 0)
#     ]
#     for secondary in secondaries.sceneName:
#         sbas_pairs.add((reference, secondary))
#         print(reference[17:25], secondary[17:25])
# print('The total number of interferometric pairs are: ', len(sbas_pairs))

## 2. Request On Demand InSAR products from ASF HyP3

Use your [NASA Earthdata login](https://urs.earthdata.nasa.gov/) to connect to [ASF HyP3](https://hyp3-docs.asf.alaska.edu/).

In [6]:
import hyp3_sdk as sdk
hyp3 = sdk.HyP3(prompt=True)

NASA Earthdata Login username:  thapa2044
NASA Earthdata Login password:  ········


In [7]:
print(sdk.version('hyp3_sdk'))

7.4.0


In [8]:
# For mintpy, the wrapped interferograms are not needed. Here, to mannually check unwrapping errors, I include the wrapped interferograms

from pathlib import Path
from typing import List, Union
from osgeo import gdal

def clip_hyp3_products_to_common_overlap(data_path: Union[str, Path], overlap: List[float]) -> None:
    """Clip all GeoTIFF files to their common overlap
    
    Args:
        data_dir:
            directory containing the GeoTIFF files to clip
        overlap:
            a list of the upper-left x, upper-left y, lower-right-x, and lower-tight y
            corner coordinates of the common overlap
    Returns: None
    """

    
    files_for_mintpy = ['_water_mask.tif', '_corr.tif', '_unw_phase.tif', '_wrapped_phase.tif', '_dem.tif', '_lv_theta.tif', '_lv_phi.tif', '_amp.tif']
    files_del = ["amp.tif","corr.tif","dem.tif","phi.tif", "theta.tif", "phase.tif", 'mask.tif']
    
    for extension in files_for_mintpy:
        for file in data_path.rglob(f'*{extension}'):
            dst_file = file.parent / f'{file.stem}_clip{file.suffix}'
            gdal.Translate(destName=str(dst_file), srcDS=str(file), projWin=overlap)
            gdal.Warp(str(dst_file), str(dst_file), dstSRS='EPSG:4326')            
            file.unlink()
            # for pattern in files_del:
            #     unneeded_files = data_path.glob(f"*{pattern}")
            #     print(unneeded_files)
            #     for file in unneeded_files:
            #         # print(file)
            #         file.unlink()


In [10]:
from pyproj import CRS, Transformer
from math import floor

def convert_wgs84_to_utm(bbox_wgs84, epsg_code=None):
    """
    Converts a bounding box from WGS84 (latitude/longitude) to a UTM zone.

    If an EPSG code is provided, it will be used for the conversion. Otherwise,
    the function automatically calculates the appropriate UTM zone from the
    center of the bounding box.

    Args:
        bbox_wgs84 (list): A list of four floats representing the bounding box
                           in the format: [ul_lon, ul_lat, lr_lon, lr_lat],
                           where 'ul' is upper-left and 'lr' is lower-right.
        epsg_code (int, optional): The EPSG code for the target UTM system.
                                   If None, the zone is calculated automatically.
                                   Defaults to None.

    Returns:
        list: A list of four floats representing the bounding box in UTM coordinates
              (easting, northing) in the same format: [ul_easting, ul_northing,
              lr_easting, lr_northing].
        int: The EPSG code used for the conversion.
    """
    # 1. Unpack the input coordinates
    ul_lon, ul_lat, lr_lon, lr_lat = bbox_wgs84

    if epsg_code is None:
        # 2. Calculate the center of the bounding box to determine the UTM zone
        center_lon = (ul_lon + lr_lon) / 2
        center_lat = (ul_lat + lr_lat) / 2

        # 3. Calculate the UTM zone number from the center longitude
        # Formula: floor((longitude + 180) / 6) + 1
        utm_zone_number = floor((center_lon + 180) / 6) + 1

        # 4. Determine if the location is in the Northern or Southern hemisphere
        # and construct the EPSG code for the target UTM zone.
        # Northern hemisphere EPSG codes are 326xx, Southern are 327xx.
        if center_lat >= 0:
            epsg_code = 32600 + utm_zone_number
        else:
            epsg_code = 32700 + utm_zone_number
        print(f"Automatically determined EPSG code: {epsg_code}")
    else:
        print(f"Using user-defined EPSG code: {epsg_code}")


    # 5. Define the source and target Coordinate Reference Systems (CRS)
    crs_wgs84 = CRS("EPSG:4326")
    crs_utm = CRS(f"EPSG:{epsg_code}")

    # 6. Create a transformer to perform the coordinate conversion
    # always_xy=True ensures the output is in (x, y) order (easting, northing)
    transformer = Transformer.from_crs(crs_wgs84, crs_utm, always_xy=True)

    # 7. Transform the upper-left and lower-right corner points
    ul_easting, ul_northing = transformer.transform(ul_lon, ul_lat)
    lr_easting, lr_northing = transformer.transform(lr_lon, lr_lat)

    # 8. Return the transformed bounding box and the EPSG code
    bbox_utm = [ul_easting, ul_northing, lr_easting, lr_northing]
    
    return bbox_utm, epsg_code

# --- Example Usage ---
if __name__ == "__main__":
    # Your original input array seems to have lat/lon values mixed.
    # A correct bounding box format is [lon_min, lat_max, lon_max, lat_min].
    # Let's assume the coordinates are for a region in Nepal.
    # Longitudes: 85.2, 85.5. Latitudes: 27.848679, 27.571514
    
    # Bounding box in WGS84 format: [upper-left lon, upper-left lat, lower-right lon, lower-right lat]
    overlap_wgs84 = [85.2, 27.848679, 85.5, 27.571514]

    print(f"Original WGS84 Bounding Box: {overlap_wgs84}")
    
    try:
        # --- Scenario 1: Automatic UTM zone calculation ---
        print("\n--- Running with Automatic Zone Detection ---")
        converted_bbox_auto, utm_epsg_auto = convert_wgs84_to_utm(overlap_wgs84)
        print(f"Calculated UTM Zone EPSG Code: {utm_epsg_auto}")
        print(f"Converted UTM Bounding Box: {converted_bbox_auto}")

        # --- Scenario 2: User-defined UTM zone ---
        print("\n--- Running with User-Defined Zone (EPSG:32645) ---")
        user_defined_epsg = 32645
        converted_bbox_user, utm_epsg_user = convert_wgs84_to_utm(overlap_wgs84, epsg_code=user_defined_epsg)
        # Note: utm_epsg_user will be the same as user_defined_epsg
        print(f"Provided UTM Zone EPSG Code: {utm_epsg_user}")
        print(f"Converted UTM Bounding Box: {converted_bbox_user}")
        
        print("\nFormat: [upper-left easting, upper-left northing, lower-right easting, lower-right northing]")

    except Exception as e:
        print(f"\nAn error occurred. Please ensure 'pyproj' is installed (`pip install pyproj`).")
        print(f"Error details: {e}")



Original WGS84 Bounding Box: [85.2, 27.848679, 85.5, 27.571514]

--- Running with Automatic Zone Detection ---
Automatically determined EPSG code: 32645
Calculated UTM Zone EPSG Code: 32645
Converted UTM Bounding Box: [322756.29598406877, 3081740.8065477666, 351927.2171535619, 3050635.5271958997]

--- Running with User-Defined Zone (EPSG:32645) ---
Using user-defined EPSG code: 32645
Provided UTM Zone EPSG Code: 32645
Converted UTM Bounding Box: [322756.29598406877, 3081740.8065477666, 351927.2171535619, 3050635.5271958997]

Format: [upper-left easting, upper-left northing, lower-right easting, lower-right northing]


In [12]:
# kathmandu
# x1,y2,x2,y1
overlap_wgs84 = [84.8,28.51,85.83,27.6] # upper-left x, upper-left y, lower-right-x, and lower-right y, x1,y2,x2,y1

In [13]:
overlap, utm_epsg_auto = convert_wgs84_to_utm(overlap_wgs84)
converted_bbox_auto

Automatically determined EPSG code: 32645


[322756.29598406877, 3081740.8065477666, 351927.2171535619, 3050635.5271958997]

In [14]:
print(data_dir)

/home/jovyan/kathmandu_asc_2018_2020_10by2


In [15]:
jobs = hyp3.find_jobs(name=project_name)
print(jobs)

273 HyP3 Jobs: 273 succeeded, 0 failed, 0 running, 0 pending.


In [16]:
succeeded_jobs = jobs.filter_jobs(succeeded=True, running=False, failed=False)
# succeeded_jobs

In [24]:
# A job is a 'Batch'
# hyp3.my_info()
# import datetime
for index in range(0, len(succeeded_jobs)):
# for index in range(0, 30):

    job = succeeded_jobs[index]
    insar_product = job.download_files(data_dir)
    product_dir = sdk.util.extract_zipped_product(insar_product[0])
    clip_hyp3_products_to_common_overlap(product_dir, overlap)

    for pattern in ["xml","png","kmz","md.txt"]:
        unneeded_files = data_dir.glob(f"*/*.{pattern}")
        for file in unneeded_files:
            file.unlink()
    # If delete the orginal interferograms
    for pattern in ["amp.tif","corr.tif","dem.tif","phi.tif", "theta.tif", "phase.tif", 'mask.tif']:
        unneeded_files = data_dir.glob(f"*/*{pattern}")
        for file in unneeded_files:
            file.unlink()    

S1AA_20201027T122222_20201202T122221_VVP036_INT40_G_ueF_844C.zip:   0%|          | 0/359701131 [00:00<?, ?it/s…

S1AA_20201120T122222_20201214T122221_VVP024_INT40_G_ueF_63E1.zip:   0%|          | 0/357760029 [00:00<?, ?it/s…

S1AA_20201027T122222_20201120T122222_VVP024_INT40_G_ueF_A065.zip:   0%|          | 0/362207099 [00:00<?, ?it/s…

S1AA_20201120T122222_20201226T122220_VVP036_INT40_G_ueF_0E29.zip:   0%|          | 0/361531379 [00:00<?, ?it/s…

S1AA_20201108T122222_20201214T122221_VVP036_INT40_G_ueF_3B57.zip:   0%|          | 0/365255853 [00:00<?, ?it/s…

...

S1AA_20180511T122202_20180523T122202_VVP012_INT40_G_ueF_641C.zip:   0%|          | 0/361620873 [00:00<?, ?it/s…

S1AA_20180511T122202_20180604T122203_VVP024_INT40_G_ueF_323F.zip:   0%|          | 0/362007410 [00:00<?, ?it/s…

S1AA_20180417T122201_20180511T122202_VVP024_INT40_G_ueF_547F.zip:   0%|          | 0/361018390 [00:00<?, ?it/s…

S1AA_20180324T122200_20180429T122201_VVP036_INT40_G_ueF_CFE5.zip:   0%|          | 0/358417097 [00:00<?, ?it/s…

S1AA_20180405T122200_20180511T122202_VVP036_INT40_G_ueF_753D.zip:   0%|          | 0/364447090 [00:00<?, ?it/s…

# 3. Cleanup Unused Files

**Delete unneeded files:**

In [25]:
print(data_dir)

/home/jovyan/kathmandu_asc_2018_2020_10by2


In [26]:
for pattern in ["xml","png","kmz","md.txt"]:
    unneeded_files = data_dir.glob(f"*/*.{pattern}")
    for file in unneeded_files:
        file.unlink()

In [27]:
# If delete the orginal interferograms
for pattern in ["amp.tif","corr.tif","dem.tif","phi.tif", "theta.tif", "phase.tif", 'mask.tif']:
    unneeded_files = data_dir.glob(f"*/*{pattern}")
    for file in unneeded_files:
        file.unlink()

In [28]:
# Do not run this step.
# files_for_mintpy = ['_water_mask_clipped_clip.tif', '_corr_clipped_clip.tif', '_unw_phase_clipped_clip.tif', '_dem_clipped.tif', '_lv_theta_clipped.tif', '_lv_phi_clipped.tif']

# import os
# data_dir = work_dir / 'insar_data'
# # for extension in files_for_mintpy:
# for file in data_dir.rglob(f'*_clipped_clip.tif'):
#     dst = str(file).replace('_clipped_clip', '_clip')
#     os.rename(file, dst)
#     # print(dst)
#     # print(file.parent / f'{file.stem}_clip{file.suffix}')
#         # os.rename(file, file.parent / f'{file.stem}_clip{file.suffix}')
#         # dst_file = file.parent / f'{file.stem}_clip{file.suffix}'